# Agentic RAG and MCP Integration on Databricks

In [0]:
%pip install databricks-vectorsearch
%pip install langchain_community
%pip install langgraph
# dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


This notebook documents the implementation of an agentic retrieval-augmented generation system built on Databricks, LangChain, LangGraph, and the Model Context Protocol (MCP). The project focuses on turning unstructured research content into a queryable knowledge base and exposing that knowledge to an autonomous agent through structured tool integration.

### Step 0: Environment and Infrastructure Setup

This section provisions the required Databricks catalog, volumes, and schemas that support data ingestion, parsing, indexing, and retrieval.

In [0]:
import os
import re
import time
from dotenv import load_dotenv
import numpy as np
import pandas as pd
from typing import Iterator
from dotenv import load_dotenv
from openai import AzureOpenAI
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import (
    ChatPromptTemplate, 
    SystemMessagePromptTemplate, 
    HumanMessagePromptTemplate
)
from databricks.vector_search.client import VectorSearchClient
from pyspark.sql.functions import udf, explode, col, pandas_udf
from pyspark.sql.types import ArrayType, StructType, StructField, StringType

load_dotenv("/Workspace/Users/26100305@lums.edu.pk/env")

# THE ABOVE IS WHAT YOU NEED FOR THIS ASSIGNMENT

True

In [0]:
from openai import AzureOpenAI

def get_client():
    return AzureOpenAI(
        api_key        = os.environ["AZURE_OPENAI_API_KEY"],
        azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
        api_version    = os.environ["AZURE_OPENAI_API_VERSION"]
    )

## Part 1: Building a RAG System from the Ground Up

The first part of the project implements a custom retrieval pipeline that transforms a research PDF into a searchable knowledge base using chunking, embeddings, and vector similarity search.

In [0]:
ROLL_NUMBER = "26100305"
CATALOG = f"{ROLL_NUMBER}-pa3"

spark.sql(f"USE CATALOG `{CATALOG}`")
print(f"✓ Using catalog: {CATALOG}")

✓ Using catalog: 26100305-pa3


### 1. Knowledge Base Construction

This stage ingests the PDF document, parses its contents, and structures the extracted information into a usable knowledge repository for downstream retrieval.

In [0]:
VOLUME_PATH = f"/Volumes/{CATALOG}/default/rag_documents/"

In [0]:
PDF_PATH = f"/Volumes/{CATALOG}/default/rag_documents/rag_research_paper.pdf"

spark.sql(f"""
    CREATE OR REPLACE TABLE `{CATALOG}`.parsed_data.parsed_documents AS
    SELECT
        '{PDF_PATH}' AS document_path,
        ai_parse_document('{PDF_PATH}') AS parsed_document
""")

display(spark.table(f"`{CATALOG}`.parsed_data.parsed_documents"))

document_path parsed_document /Volumes/26100305-pa3/default/rag_documents/rag_research_paper.pdf {"document":{"elements":[{"bbox":[{"coord":[305,270,1395,385],"page_id":0}],"confidence":0.9924,"content":"Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents","description":null,"id":0,"type":"title"},{"bbox":[{"coord":[448,494,1252,533],"page_id":0}],"confidence":0.9828,"content":"Tiannuo Yang1*, Zebin Yao1*, Bowen Jin2, Lixiao Cui1, Yusen Li1,","description":null,"id":1,"type":"text"},{"bbox":[{"coord":[662,530,1030,565],"page_id":0}],"confidence":0.998,"content":"Gang Wang1, Xiaoguang Liu1","description":null,"id":2,"type":"text"},{"bbox":[{"coord":[725,562,968,594],"page_id":0}],"confidence":0.9999,"content":"1 Nankai University","description":null,"id":3,"type":"text"},{"bbox":[{"coord":[603,592,1093,629],"page_id":0}],"confidence":0.9991,"content":"2 University of Illinois Urbana-Champaign","description":null,"id":4,"type":"text"},{"bbox":[{"coord":[416,626,1284,658],"page_id":0}],"confidence":0.9714,"content":"{yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn","description":null,"id":5,"type":"text"},{"bbox":[{"coord":[696,656,1000,690],"page_id":0}],"confidence":0.9994,"content":"bowenj4@illinois.edu","description":null,"id":6,"type":"text"},{"bbox":[{"coord":[780,761,920,803],"page_id":0}],"confidence":0.9996,"content":"Abstract","description":null,"id":7,"type":"section_header"},{"bbox":[{"coord":[389,832,1313,1390],"page_id":0}],"confidence":0.9983,"content":"Large Language Model (LLM)-based search agents have shown remarkable ca-\npabilities in solving complex tasks by dynamically decomposing problems and\naddressing them through interleaved reasoning and retrieval. However, this in-\nterleaved paradigm introduces substantial efficiency bottlenecks. First, we ob-\nserve that both highly accurate and overly approximate retrieval methods de-\ngrade system efficiency: exact search incurs significant retrieval overhead, while\ncoarse retrieval requires additional reasoning steps during generation. Second,\nwe identify inefficiencies in system design, including improper scheduling and\nfrequent retrieval stalls, which lead to cascading latency—where even minor de-\nlays in retrieval amplify end-to-end inference time. To address these challenges,\nwe introduce SearchAgent-X, a high-efficiency inference framework for LLM-\nbased search agents. SearchAgent-X leverages high-recall approximate retrieval\nand incorporates two key techniques: priority-aware scheduling and non-stall\nretrieval. Extensive experiments demonstrate that SearchAgent-X consistently\noutperforms state-of-the-art systems such as vLLM and HNSW-based retrieval\nacross diverse tasks, achieving up to 3.4× higher throughput and 5× lower la-\ntency, without compromising generation quality. SearchAgent-X is available at\nhttps://github.com/tiannuo-yang/SearchAgent-X.","description":null,"id":8,"type":"text"},{"bbox":[{"coord":[290,1441,537,1483],"page_id":0}],"confidence":0.9996,"content":"1 Introduction","description":null,"id":9,"type":"section_header"},{"bbox":[{"coord":[290,1510,1410,1792],"page_id":0}],"confidence":0.9975,"content":"Traditional Retrieval-Augmented Generation (RAG) typically uses a sequential retrieve-then-generate approach [1–8], which limits dynamic interaction with knowledge bases. Recent advancements have ushered in RAG 2.0, known as Search Agents [9–16]. This paradigm leverages the strong reasoning capabilities of Large Language Models (LLMs), allowing for the dynamic and adaptive interleaving of reasoning steps with retrieval calls throughout the generation process. Instead of a fixed pipeline, search agents can decide when and what to retrieve based on LLM’s ongoing reasoning, leading to significant improvements in the quality and depth of the generated responses. Leveraging post-training techniques similar to DeepSeek-R1, some pioneering models can even autonomously initiate retrieval action

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE `{CATALOG}`.knowledge_base_data.knowledge_base (
        Id BIGINT GENERATED ALWAYS AS IDENTITY,
        Title   STRING,
        Authors STRING,
        Content STRING,
        DocumentURI STRING
    )
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

DataFrame[]

In [0]:
spark.sql(f"""
    INSERT INTO `{CATALOG}`.knowledge_base_data.knowledge_base
        (Title, Authors, Content, DocumentURI)
    SELECT
        ai_extract(LEFT(CAST(parsed_document AS STRING), 50000), array('title')).title     AS Title,
        ai_extract(LEFT(CAST(parsed_document AS STRING), 50000), array('authors')).authors AS Authors,
        concat_ws(' ', transform(
            try_cast(parsed_document:document:elements AS ARRAY<VARIANT>),
            e -> try_cast(e:content AS STRING)
        )) AS Content,
        document_path AS DocumentURI
    FROM `{CATALOG}`.parsed_data.parsed_documents
""")

display(spark.table(f"`{CATALOG}`.knowledge_base_data.knowledge_base"))

Id Title Authors Content DocumentURI 1 Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents ["Tiannuo Yang","Zebin Yao","Bowen Jin","Lixiao Cui","Yusen Li","Gang Wang","Xiaoguang Liu"] Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents Tiannuo Yang1*, Zebin Yao1*, Bowen Jin2, Lixiao Cui1, Yusen Li1, Gang Wang1, Xiaoguang Liu1 1 Nankai University 2 University of Illinois Urbana-Champaign {yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn bowenj4@illinois.edu Abstract Large Language Model (LLM)-based search agents have shown remarkable ca-
pabilities in solving complex tasks by dynamically decomposing problems and
addressing them through interleaved reasoning and retrieval. However, this in-
terleaved paradigm introduces substantial efficiency bottlenecks. First, we ob-
serve that both highly accurate and overly approximate retrieval methods de-
grade system efficiency: exact search incurs significant retrieval overhead, while
coarse retrieval requires additional reasoning steps during generation. Second,
we identify inefficiencies in system design, including improper scheduling and
frequent retrieval stalls, which lead to cascading latency—where even minor de-
lays in retrieval amplify end-to-end inference time. To address these challenges,
we introduce SearchAgent-X, a high-efficiency inference framework for LLM-
based search agents. SearchAgent-X leverages high-recall approximate retrieval
and incorporates two key techniques: priority-aware scheduling and non-stall
retrieval. Extensive experiments demonstrate that SearchAgent-X consistently
outperforms state-of-the-art systems such as vLLM and HNSW-based retrieval
across diverse tasks, achieving up to 3.4× higher throughput and 5× lower la-
tency, without compromising generation quality. SearchAgent-X is available at
https://github.com/tiannuo-yang/SearchAgent-X. 1 Introduction Traditional Retrieval-Augmented Generation (RAG) typically uses a sequential retrieve-then-generate approach [1–8], which limits dynamic interaction with knowledge bases. Recent advancements have ushered in RAG 2.0, known as Search Agents [9–16]. This paradigm leverages the strong reasoning capabilities of Large Language Models (LLMs), allowing for the dynamic and adaptive interleaving of reasoning steps with retrieval calls throughout the generation process. Instead of a fixed pipeline, search agents can decide when and what to retrieve based on LLM’s ongoing reasoning, leading to significant improvements in the quality and depth of the generated responses. Leveraging post-training techniques similar to DeepSeek-R1, some pioneering models can even autonomously initiate retrieval actions during reasoning without intermediate supervision [12–14]. However, the improved generation quality achieved by search agents often comes at the cost of
efficiency—an overhead that is nontrivial in practical deployments. In reasoning-with-search sce-
narios, achieving low-latency responses is critical for ensuring a seamless user experience [17, 18].
Moreover, during post-training of LLM-based search agents, efficient model rollouts over large-scale *Equal Contribution Preprint. Under review. arXiv:2505.12065v1 [cs.AI] 17 May 2025 training corpora are essential to support scalable learning. While recent systems incorporate advanced
inference optimizations—such as sequence concatenation [12–14] and prefix caching [18–20]—these
techniques are not specifically designed to address the unique computational challenges posed by the
tight interleaving of multi-step reasoning and dynamic retrieval. To this end, we first conduct a systematic analysis of the efficiency factors governing LLM-based search agents, uncovering insights that diverge from the understanding of naive RAG. Our in-depth analysis reveals two key observations: First, we demonstrate a non-monotonic relationship between retrieval accuracy and end-to-end efficiency. Both excessively hig

In [0]:
kb_text = spark.table(f"`{CATALOG}`.knowledge_base_data.knowledge_base") \
               .collect()[0]["Content"]
print(f"✓ Content: {len(kb_text)} characters")

✓ Content: 63995 characters


In [0]:
raw_size = spark.sql(f"""
    SELECT length(CAST(parsed_document AS STRING)) AS raw_length
    FROM `{CATALOG}`.parsed_data.parsed_documents
""").collect()[0]["raw_length"]

print(f"Raw parsed document: {raw_size} characters")
print(f"Extracted content:   {len(kb_text)} characters")
print(f"Difference:          {raw_size - len(kb_text)} characters")

Raw parsed document: 91237 characters
Extracted content:   63995 characters
Difference:          27242 characters


In [0]:
# Check beginning, middle and end of content
print("=== BEGINNING ===")
print(kb_text[:300])
print("\n=== MIDDLE ===")
mid = len(kb_text) // 2
print(kb_text[mid:mid+300])
print("\n=== END ===")
print(kb_text[-300:])

=== BEGINNING ===
Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents Tiannuo Yang1*, Zebin Yao1*, Bowen Jin2, Lixiao Cui1, Yusen Li1, Gang Wang1, Xiaoguang Liu1 1 Nankai University 2 University of Illinois Urbana-Champaign {yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn 

=== MIDDLE ===
red by semantic similarity. Second, 8 End-to-End Latency Prefix Cache Hit Rate Waiting + Prefill Decode Retrieval Terminate Ratio = 24% 0.16s 0.15s vLLM_ANN + Prefix Cache + Priority Scheduling + Non-Stall Retrieval vLLM_ENN vLLM_ANN + Prefix Cache + Priority + Non-Stall Scheduling Retrieval Figure 

=== END ===
g term, this line of work may support practical search agents in real-world deployments, such as OpenAI’s DeepResearch [16] and xAI’s DeepSearch [15], thereby improving fair information access and generating economic value. We are not aware of any negative societal impacts arising from this work. 18


### 2. Chunking Strategies for Context Retrieval

Two chunking approaches are compared here: a fixed-size splitter for stable context windows and a semantic splitter for preserving topic coherence across related sentences.

In [0]:
class MyTextSplitter():
    def __init__(self, chunk_size=512, chunk_overlap=0):
        ### TO-DO: Implement the constructor for MyTextSplitter
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
    
    def split_text(self, text):
        chunks = []
        ### TO-DO: Implement the split_text method
        start = 0
        while start < len(text):
            end = start + self.chunk_size
            chunks.append(text[start:end])
            start += self.chunk_size - self.chunk_overlap
        return chunks


class MySemanticSplitter:
    def __init__(self, client, model_name, distance_threshold=0.3, buffer_size=1):
        ### TO-DO: Implement the constructor for MySemanticSplitter
        self.client = client
        self.model_name = model_name
        self.distance_threshold = distance_threshold
        self.buffer_size = buffer_size
    
    def _combine_sentences(self, sentences, buffer_size):
        combined = []
        ### TO-DO: Implement the _combine_sentences method
        for i in range(len(sentences)):
            start = max(0, i - buffer_size)
            end = min(len(sentences), i + buffer_size + 1)
            combined.append(" ".join(sentences[start:end]))
        return combined

    @staticmethod
    def cosine_similarity(embeddings1, embeddings2):
        ### TO-DO: Implement the cosine_similarity method
        a = np.array(embeddings1)
        b = np.array(embeddings2)
        cosine_similarities = np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b))
        return cosine_similarities

    def _get_embedding(self, text):
        ### TO-DO: Implement the _get_embedding method
        response = self.client.embeddings.create(input=[text], model=self.model_name)
        embedding = response.data[0].embedding
        return embedding

    def split_text(self, text):
        chunks = []
        ### TO-DO: Implement the split_text method

        # Step 1: split into sentences
        sentences = text.split(". ")
        sentences = [s.strip() for s in sentences if s.strip()]

        if len(sentences) == 0:
            return chunks

        # Step 2: pad each sentence with its neighbors
        combined = self._combine_sentences(sentences, self.buffer_size)

        # Step 3: embed every padded sentence
        embeddings = [self._get_embedding(s) for s in combined]

        # Step 4 & 5: compute distances between consecutive embeddings
        # distance = 1 - cosine_similarity
        # if distance > threshold → cut here → new chunk starts
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            similarity = self.cosine_similarity(embeddings[i-1], embeddings[i])
            distance   = 1 - similarity

            if distance > self.distance_threshold:
                # meaning has shifted — save current chunk, start new one
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]
            else:
                # meaning is still close — keep adding to current chunk
                current_chunk.append(sentences[i])

        # don't forget the last chunk
        if current_chunk:
            chunks.append(" ".join(current_chunk))
            
        return chunks
    

In [0]:
### SANITY CHECK ###

# Sample Research Paper
sample_document = (
    "Abstract: The exponential growth in data has created a need for platforms capable of storing both structured & unstructured data, effectively processing the data, analyzing, and creating machine learning models. Traditional data lakes and warehouses often lack the flexibility and performance to provide these capabilities. So, the new Lakehouse paradigm is introduced. Databricks is an implementation of the Lakehouse paradigm that is cloud-native and built upon Apache Spark. However, there is a lack of substantial academic work describing the ecosystem. This paper presents a comprehensive description of the Databricks ecosystem, showing it both as an architecture and as a platform already in use. We will go through the main components of Databricks architecture, including Delta Lake, Unity Catalog, cluster management, Azure integrations, and discuss their roles in creating secure, scalable, and cost-effective data engineering workflows. We also present an experiment that is designed to bridge the gap between theory and practice by demonstrating the ingestion of data, feature engineering, and basic fraud detection. We utilized a synthetic dataset of financial transactions. The experimental procedure and metrics such as query latency, processing throughput, speed, and performance results are described in detail offering a reproducible benchmark for evaluating similar workloads in Databricks. This work is designed to function as a technical resource for individuals in industry, data engineers, and researchers who are interested in working with the Databricks environment for large-scale analytics. Keywords: Databricks, Lakehouse, Delta Lake, Data Engineering, PySpark, Unity Catalog, MLflow, Azure Synapse, Data Factory, Power BI, Real-time Analytics, Cloud Data Platforms, Fraud Detection, Performance Benchmarking, Industry Use Case. 1. Introduction These days organizations have high volume of data in the form of structured, semistructured and unstructured data. They need platforms that can integrate storage, analytics and machine learning at scale in order to gain value from their data. Conventional data warehouses store structured data for analytical needs (such as reporting through Power BI), but lack flexibility. Data lakes, on the other hand, can store semistructured and unstructured data using batch data processing, but do not provide ACID guarantees. The new Lakehouse model provides the best of both worlds to store and..."
)

# --- Test 1: The Normal Text Splitter (Fixed Size) ---
print("====== FIXED-SIZE CHUNKER ======")
normal_splitter = MyTextSplitter(chunk_size=512, chunk_overlap=50)
normal_chunks = normal_splitter.split_text(sample_document)

for i, chunk in enumerate(normal_chunks[:3]):
    print(f"\n--- Chunk {i+1} (Length: {len(chunk)}) ---")
    print(chunk)


# --- Test 2: The Semantic Splitter (Embedding-Based) ---
print("\n\n====== SEMANTIC CHUNKER ======")
semantic_splitter = MySemanticSplitter(
    client=get_client(), 
    model_name=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"], 
    distance_threshold=0.15, 
    buffer_size=1
)
semantic_chunks = semantic_splitter.split_text(sample_document)

for i, chunk in enumerate(semantic_chunks):
    print(f"\n--- Chunk {i+1} (Length: {len(chunk)}) ---")
    print(chunk)

====== FIXED-SIZE CHUNKER ======

--- Chunk 1 (Length: 512) ---
Abstract: The exponential growth in data has created a need for platforms capable of storing both structured & unstructured data, effectively processing the data, analyzing, and creating machine learning models. Traditional data lakes and warehouses often lack the flexibility and performance to provide these capabilities. So, the new Lakehouse paradigm is introduced. Databricks is an implementation of the Lakehouse paradigm that is cloud-native and built upon Apache Spark. However, there is a lack of substa

--- Chunk 2 (Length: 512) ---
n Apache Spark. However, there is a lack of substantial academic work describing the ecosystem. This paper presents a comprehensive description of the Databricks ecosystem, showing it both as an architecture and as a platform already in use. We will go through the main components of Databricks architecture, including Delta Lake, Unity Catalog, cluster management, Azure integrations, and di

In [0]:
from pyspark.sql import Row

fixed_splitter = MyTextSplitter(chunk_size=512, chunk_overlap=50)

fixed_chunks = fixed_splitter.split_text(kb_text)
fixed_df = spark.createDataFrame([Row(ChunkText=c) for c in fixed_chunks])
fixed_df.write.mode("overwrite").saveAsTable(f"`{CATALOG}`.processed_data.fixed_chunks")
print(f"fixed_chunks: {len(fixed_chunks)} chunks saved")
display(fixed_df.limit(3))


fixed_chunks: 139 chunks saved


ChunkText
"Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents Tiannuo Yang1*, Zebin Yao1*, Bowen Jin2, Lixiao Cui1, Yusen Li1, Gang Wang1, Xiaoguang Liu1 1 Nankai University 2 University of Illinois Urbana-Champaign {yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn bowenj4@illinois.edu Abstract Large Language Model (LLM)-based search agents have shown remarkable ca- pabilities in solving complex tasks by dynamically decomposing problems and addressing them through interleav"
"ing problems and addressing them through interleaved reasoning and retrieval. However, this in- terleaved paradigm introduces substantial efficiency bottlenecks. First, we ob- serve that both highly accurate and overly approximate retrieval methods de- grade system efficiency: exact search incurs significant retrieval overhead, while coarse retrieval requires additional reasoning steps during generation. Second, we identify inefficiencies in system design, including improper scheduling and frequent retrieva"
"ncluding improper scheduling and frequent retrieval stalls, which lead to cascading latency—where even minor de- lays in retrieval amplify end-to-end inference time. To address these challenges, we introduce SearchAgent-X, a high-efficiency inference framework for LLM- based search agents. SearchAgent-X leverages high-recall approximate retrieval and incorporates two key techniques: priority-aware scheduling and non-stall retrieval. Extensive experiments demonstrate that SearchAgent-X consistently outperfor"


In [0]:
client = AzureOpenAI(
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
)

semantic_splitter = MySemanticSplitter(
    client=client,
    model_name=os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    distance_threshold=0.15,
    buffer_size = 1
)

# 1. Load the knowledge base table
kb_text = spark.table(f"`{CATALOG}`.knowledge_base_data.knowledge_base").collect()[0]["Content"]

# 2. Split the content into chunks
semantic_chunks = semantic_splitter.split_text(kb_text)
print(f"Semantic chunks: {len(semantic_chunks)} chunks")

# 3. Write the chunks to a new table
semantic_df = spark.createDataFrame([Row(ChunkText=c) for c in semantic_chunks])
semantic_df.write.mode("overwrite").saveAsTable(f"`{CATALOG}`.processed_data.semantic_chunks")
print(f"✓ Saved to processed_data.semantic_chunks")
display(semantic_df.limit(3))


Semantic chunks: 309 chunks
✓ Saved to processed_data.semantic_chunks


ChunkText
"Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents Tiannuo Yang1*, Zebin Yao1*, Bowen Jin2, Lixiao Cui1, Yusen Li1, Gang Wang1, Xiaoguang Liu1 1 Nankai University 2 University of Illinois Urbana-Champaign {yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn bowenj4@illinois.edu Abstract Large Language Model (LLM)-based search agents have shown remarkable ca- pabilities in solving complex tasks by dynamically decomposing problems and addressing them through interleaved reasoning and retrieval However, this in- terleaved paradigm introduces substantial efficiency bottlenecks"
"First, we ob- serve that both highly accurate and overly approximate retrieval methods de- grade system efficiency: exact search incurs significant retrieval overhead, while coarse retrieval requires additional reasoning steps during generation"
"Second, we identify inefficiencies in system design, including improper scheduling and frequent retrieval stalls, which lead to cascading latency—where even minor de- lays in retrieval amplify end-to-end inference time To address these challenges, we introduce SearchAgent-X, a high-efficiency inference framework for LLM- based search agents SearchAgent-X leverages high-recall approximate retrieval and incorporates two key techniques: priority-aware scheduling and non-stall retrieval Extensive experiments demonstrate that SearchAgent-X consistently outperforms state-of-the-art systems such as vLLM and HNSW-based retrieval across diverse tasks, achieving up to 3.4× higher throughput and 5× lower la- tency, without compromising generation quality"


### 3. Embedding Generation for Semantic Search

The chunked text is converted into vector embeddings so that relevant passages can be retrieved through similarity-based search.

In [0]:
class GenerateEmbeddings:
    def __init__(self, client, model_name, chunk_table_name):
        ### TO-DO: Implement the __init__ method
        self.client = client
        self.model_name = model_name
        self.chunk_table_name = chunk_table_name
        self.embeddings = None

    def embed(self):
        records = []
        ### TO-DO: Implement the embed method
        records = []
        chunks_df = spark.table(self.chunk_table_name).collect()
        
        for row in chunks_df:
            chunk_text = row["ChunkText"]
            response = self.client.embeddings.create(
                input=[chunk_text],
                model=self.model_name
            )
            embedding = response.data[0].embedding
            
            records.append(Row(
                ChunkText = chunk_text,
                Embedding = embedding
            ))
        
        self.embeddings = spark.createDataFrame(records)

    def save_to_table(self, table_name):
        ### DONE: Saves the embeddings to a Delta table
        self.embeddings.write.format("delta").mode("overwrite").saveAsTable(table_name)



In [0]:
### TO-DO: Store the Fixed-Size Embeddings in a Delta Table
fixed_embeddings = GenerateEmbeddings(
    client           = get_client(),
    model_name       = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    chunk_table_name = f"`{CATALOG}`.processed_data.fixed_chunks"
)
fixed_embeddings.embed()
fixed_embeddings.save_to_table(f"`{CATALOG}`.processed_data.fixed_embeddings")
display(spark.table(f"`{CATALOG}`.processed_data.fixed_embeddings").limit(3))

ChunkText Embedding Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents Tiannuo Yang1*, Zebin Yao1*, Bowen Jin2, Lixiao Cui1, Yusen Li1, Gang Wang1, Xiaoguang Liu1 1 Nankai University 2 University of Illinois Urbana-Champaign {yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn bowenj4@illinois.edu Abstract Large Language Model (LLM)-based search agents have shown remarkable ca-
pabilities in solving complex tasks by dynamically decomposing problems and
addressing them through interleav List(0.00552639365196228, 0.002423459431156516, 0.046240512281656265, 0.013432458974421024, -0.01787773333489895, 0.024231575429439545, -0.029232509434223175, 0.04611971974372864, -0.03635944426059723, 0.03391937538981438, 0.03645607829093933, -0.06421488523483276, -0.05242524296045303, 0.010038105770945549, -0.010237418115139008, 0.0019568868447095156, 0.0050643510185182095, -0.019460152834653854, 0.03172089532017708, 0.016102038323879242, 0.05300506204366684, -0.02269747294485569, -0.002109391149133444, 0.026840757578611374, 0.028773486614227295, -0.0060669537633657455, 0.01898905076086521, 0.03411264717578888, 0.014724970795214176, -0.028531894087791443, 0.026937395334243774, -0.0234222449362278, -0.012683526612818241, 0.0246664397418499, -0.03099612332880497, 0.05856165289878845, 0.01511151622980833, 0.007386644370853901, -0.01994333602488041, 0.004572109319269657, 0.005837441887706518, 0.006933661177754402, -0.007374564651399851, -0.00484993914142251, 0.010418610647320747, 0.03176921233534813, -0.001662447932176292, 0.01939975656569004, -0.010605843737721443, 0.003877535229548812, 0.012659367173910141, -0.0034547511022537947, 0.020619791001081467, -0.06025278940796852, -0.024980507791042328, -0.03983835130929947, 0.0202211644500494, 0.0040285298600792885, 0.021586153656244278, 0.015643015503883362, -0.04549158364534378, -0.02778296358883381, -0.006275325547903776, 0.05928642675280571, -0.044597696512937546, -0.04529830813407898, -0.040369853377342224, -0.013649890199303627, -0.029788168147206306, -0.006933661177754402, -0.026115985587239265, -0.0014555857051163912, 0.05242524296045303, -0.0015929905930534005, 0.006462558638304472, 0.006275325547903776, -0.0022528360132128, 0.06387665867805481, -0.02154991589486599, 0.0010916892206296325, -4.7940711374394596E-4, 0.01414515171200037, 0.024183256551623344, -0.0678870677947998, -0.03655271604657173, -0.016899289563298225, -0.0017258655279874802, 0.013347901403903961, -0.012405697256326675, -0.05899651721119881, -0.02778296358883381, 0.031237713992595673, -0.015461822971701622, -0.009138179011642933, 0.06025278940796852, 0.042809922248125076, -0.016766414046287537, 0.01176548097282648, 0.07547302544116974, 0.05455124378204346, -0.002476307563483715, -0.020643949508666992, 0.04309983178973198, -0.04382460564374924, -0.006384041626006365, -0.02054731361567974, -0.018324676901102066, -0.035368919372558594, -0.03324291855096817, 0.008334889076650143, -0.09354402869939804, 0.010509207844734192, -0.002810005098581314, -0.015328947454690933, -0.007893985137343407, -0.056097425520420074, -0.010346134193241596, 0.0069397008046507835, -0.019919175654649734, 0.04194019362330437, 0.00942204799503088, 0.016271153464913368, -0.02003997191786766, -0.006571274716407061, 0.02113921009004116, -0.0029640193097293377, 0.020148687064647675, -0.040514808148145676, -0.0039107538759708405, 0.008854309096932411, 0.004656665958464146, -0.015715492889285088, 0.0214653592556715, -0.004976774100214243, -0.08648957312107086, -0.008340928703546524, -0.04976774379611015, 0.008365088142454624, -0.01628323271870613, 0.031165236607193947, 0.036987580358982086, -0.023373927921056747, -0.020571472123265266, 0.04884969815611839, -0.06199224665760994, 0.006794746499508619, -0.001084139570593834, -0.015328947454690933, -0.017708618193864822, 0.004191603511571884, 0.007060496602207422, -0.09992203116416931, -0.05213533341884613, 0.009742156602442265, -0.02168279141187668, -0.032759737223386765,

In [0]:
### TO-DO: Store the Semantic Embeddings in a Delta Table
semantic_embeddings = GenerateEmbeddings(
    client           = get_client(),
    model_name       = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    chunk_table_name = f"`{CATALOG}`.processed_data.semantic_chunks"
)
semantic_embeddings.embed()
semantic_embeddings.save_to_table(f"`{CATALOG}`.processed_data.semantic_embeddings")
display(spark.table(f"`{CATALOG}`.processed_data.semantic_embeddings").limit(3))

ChunkText Embedding Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents Tiannuo Yang1*, Zebin Yao1*, Bowen Jin2, Lixiao Cui1, Yusen Li1, Gang Wang1, Xiaoguang Liu1 1 Nankai University 2 University of Illinois Urbana-Champaign {yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn bowenj4@illinois.edu Abstract Large Language Model (LLM)-based search agents have shown remarkable ca-
pabilities in solving complex tasks by dynamically decomposing problems and
addressing them through interleaved reasoning and retrieval However, this in-
terleaved paradigm introduces substantial efficiency bottlenecks List(0.010479713790118694, 0.0058197020553052425, 0.05887097492814064, 0.015507061965763569, -0.01617494598031044, 0.03774154186248779, -0.025743909180164337, 0.03890730440616608, -0.03985448554158211, 0.03358851373195648, 0.04150598123669624, -0.06984856724739075, -0.0426960289478302, 0.017498571425676346, -0.011038308031857014, -0.011451181955635548, 0.014912037178874016, -0.03159700334072113, 0.0223680566996336, 0.02269592694938183, 0.06270827353000641, -0.020412975922226906, 0.0013319740537554026, 0.022149475291371346, 0.026691090315580368, -0.007953896187245846, 0.015968509018421173, 0.03431711718440056, 0.011536185629665852, -0.030674109235405922, 0.02970264106988907, -0.023655252531170845, -0.011997632682323456, 0.01783858612179756, -0.026472508907318115, 0.05935670807957649, 0.014256296679377556, -0.001023077522404492, -0.011645475402474403, 0.01013362780213356, 0.012076565064489841, 0.013564125634729862, -0.012762663885951042, -0.0011718336027115583, 0.022647352889180183, 0.04155455529689789, 0.0023467030841857195, 0.02020653896033764, -0.014851320534944534, 0.009368596598505974, 0.011414751410484314, -0.008044971153140068, 0.0056891608983278275, -0.057219479233026505, -0.021141577512025833, -0.04024307057261467, 0.022015899419784546, -0.006429905537515879, 0.01961151510477066, 0.023776685819029808, -0.0439346507191658, -0.029945509508252144, -0.010006123222410679, 0.054499369114637375, -0.04784481227397919, -0.04993346706032753, -0.03302992135286331, -0.007055288180708885, -0.037012938410043716, -0.0035549665335565805, -0.03057696297764778, -0.004708585329353809, 0.06261112540960312, -2.402107056695968E-4, 0.002755023306235671, 0.0033485295716673136, 0.008494275622069836, 0.06911996752023697, -0.013151251710951328, 0.0062963287346065044, 0.00839712843298912, 0.006587768904864788, 0.029119759798049927, -0.0688285231590271, -0.025258174166083336, -0.008554992265999317, 0.005273250862956047, 0.014827034436166286, -0.0218337494879961, -0.06290256977081299, -0.028731172904372215, 0.03460855782032013, -0.010236846283078194, -0.01234371867030859, 0.05794807896018028, 0.04714049771428108, -0.011857984587550163, 0.01088651642203331, 0.0799032598733902, 0.048451978713274, -0.013005531392991543, -0.02030368708074093, 0.04116596654057503, -0.04206457361578941, -0.012665517628192902, -0.023424528539180756, -0.021141577512025833, -0.027201110497117043, -0.03368566185235977, 0.009787542745471, -0.08888934552669525, 0.005549512337893248, -0.004468753933906555, -0.027346830815076828, -0.010929017327725887, -0.06329115480184555, -0.019016491249203682, 0.021748745813965797, -0.015482774935662746, 0.04709192365407944, 0.0121129946783185, 0.013017674908041954, -0.021008001640439034, -0.006344901863485575, 0.018263602629303932, -0.0024909053463488817, 0.024225989356637, -0.04077737778425217, 0.00534611102193594, 0.008724999614059925, 0.010497928597033024, -0.01879790984094143, 0.017753582447767258, 0.0028764568269252777, -0.08718927204608917, -0.01551920548081398, -0.0612996444106102, 0.009089299477636814, -0.008828218095004559, 0.024638863280415535, 0.02737111784517765, -0.01984223909676075, -0.017899302765727043, 0.03873729705810547, -0.06557410955429077, 0.01623566262423992, -0.005476652178913355, -0.021724458783864975, 4.284326860215515E-4, 0.0017638220451772213, 0.005491831339895725, -0.09967264533042908

### 4. Vector Search and Retrieval Evaluation

This section implements a lightweight similarity search mechanism and evaluates which retrieval strategy provides the most relevant context for question answering.

In [0]:
class VectorSearch:
    def __init__(self, client, model_name, embeddings_table):
        ### TO-DO: Implement the __init__ method
        self.client = client
        self.model_name = model_name
        self.embeddings_table = embeddings_table

    def search_vectors(self, query, top_k=3):
        ### TO-DO: Implement the search_vectors method
        ###     1. Embed the query
        response    = self.client.embeddings.create(
            input=[query],
            model=self.model_name
        )
        query_vec = np.array(response.data[0].embedding)

        ###     2. Calculate the cosine similarity between the query and each embedding
        rows    = spark.table(self.embeddings_table).collect()
        results = []
        for row in rows:
            chunk_vec  = np.array(row["Embedding"])
            similarity = float(
                np.dot(query_vec, chunk_vec) /
                (np.linalg.norm(query_vec) * np.linalg.norm(chunk_vec))
            )
            results.append({
                "Similarity": similarity,
                "Chunk_text": row["ChunkText"]
            })
        ###     3. Sort the embeddings by similarity and return the top_k
        results = sorted(results, key=lambda x: x["Similarity"], reverse=True)
        return spark.createDataFrame(results[:top_k])
    

In [0]:
vs_fixed   = VectorSearch(
    client = get_client(),
    model_name = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    embeddings_table = f"`{CATALOG}`.processed_data.fixed_embeddings"
)
results_1 = vs_fixed.search_vectors("What is SearchAgent-X?")
display(results_1)


Chunk_text,Similarity
"insights, we propose SearchAgent-X, an inference system explicitly designed to optimize end-to-end efficiency for search agent workloads by smoothly interleaving self-reasoning and retrieval. Figure 3 shows SearchAgent-X's architecture, a tightly integrated system processing search agent requests at the token generation level. At each LLM output step, the system checks for special tags that trigger the Retriever for an ANN-based search (e.g., ) or request completion (e.g., ), respectively. To",0.6516570878588396
"e frequent retrieval stalls, SearchAgent-X proposes a non-stall retrieval mechanism through an adaptive mechanism that allows generation to proceed without unnecessary waiting while ensuring sufficient retrieval quality. Our extensive experiments demonstrate that SearchAgent-X consistently and significantly outper- forms state-of-the-art baseline systems across various operational settings. In both offline and online inference scenarios, SearchAgent-X achieves substantial improvements in system performance",0.6132522902077334
"n retrieval actions and iteratively incorporate new information into its reasoning process allows the search agent to tackle more complex questions and deliver better responses, moving beyond reliance solely on pre-trained knowledge or a single retrieval. B Implementation Details B.1 SearchAgent-X Execution This section outlines the high-level execution flow of the SearchAgent-X system, as depicted in Algorithm 1, complementing the conceptual component descriptions in Section 3 of the main paper. SearchAgen",0.5970792626911952


In [0]:
vs_semantic = VectorSearch(
    client = get_client(),
    model_name= os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    embeddings_table = f"`{CATALOG}`.processed_data.semantic_embeddings"
)
results_2 = vs_semantic.search_vectors("What is SearchAgent-X?")
display(results_2)

Chunk_text,Similarity
SearchAgent-X is available at https://github.com/tiannuo-yang/SearchAgent-X,0.7814773717570537
SearchAgent-X is designed to optimize end-to-end system throughput and latency by smoothly coordinating the interleaving of self-reasoning and retrieval,0.6220193976976989
"This ability to autonomously plan retrieval actions and iteratively incorporate new information into its reasoning process allows the search agent to tackle more complex questions and deliver better responses, moving beyond reliance solely on pre-trained knowledge or a single retrieval B Implementation Details B.1 SearchAgent-X Execution This section outlines the high-level execution flow of the SearchAgent-X system, as depicted in Algorithm 1, complementing the conceptual component descriptions in Section 3 of the main paper. SearchAgent-X orchestrates LLM inference (with prefix caching, Section 3.1), dynamic high-recall approximate retrieval, Priority Scheduling (Section 3.2), and Non-Stall Retrieval (Section 3.3) to achieve efficient search agents The system initializes an LLM inference engine and manages incoming requests, active asynchronous search tasks, and their results",0.6086752030213042


## Observations from Retrieval Comparison

The fixed-size and semantic chunking pipelines both retrieve context that supports accurate answer generation. Fixed chunking tends to preserve local structure and produce concise passages, while semantic chunking captures broader topical continuity and often yields richer context for conceptual questions.

### 5. End-to-End RAG Pipeline

This stage connects the retrieved context to an LLM so the system can answer questions grounded in the source material rather than relying on the model’s general knowledge alone.

In [0]:

llm = AzureChatOpenAI(
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT"), 
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"), 
    temperature=0.0
)

### TO-DO
system_message = SystemMessagePromptTemplate.from_template(
    "You are a precise question-answering, helpful assistant. Your ONLY source of truth is the context below.\n\n"
    "Rules you must follow without exception:\n"
    "1. Answer ONLY using information explicitly stated in the context.\n"
    "2. Do NOT use your general knowledge or make inferences beyond the context.\n"
    "3. If the context does not contain enough information to answer, respond EXACTLY with: 'I do not know'\n"
    "4. Do not make up facts, figures, or details.\n"
    "5. Provide a thorough and detailed answer using ALL relevant information from the context.\n\n"
    "Context:\n{context}"
)
### TO-DO
human_message = HumanMessagePromptTemplate.from_template("{question}")

### TO-DO
chat_prompt = ChatPromptTemplate.from_messages([system_message, human_message])

class RAGPipeline:
    def __init__(self, llm, chat_prompt, vector_search):
        ### TO-DO: Implement the __init__ method
        self.llm = llm
        self.chat_prompt = chat_prompt
        self.vector_search = vector_search
    
    def get_context(self, question):
        ### TO-DO: Implement the get_context method. Join each chunks with 2 newline seperators
        results = self.vector_search.search_vectors(question, top_k=5)
        chunks = [row["Chunk_text"] for row in results.collect()]
        return "\n\n".join(chunks)

    def invoke(self, question):
        content = None
        ### TO-DO: Implement the invoke method
        context  = self.get_context(question)
        prompt   = self.chat_prompt.format_messages(
            context  = context,
            question = question
        )
        response = self.llm.invoke(prompt)
        content  = response.content
        return content

    def invoke_w_lcel(self, question):
        ### TO-DO: Implement the same functionality but use the LCEL
        chain = (
            {
                "context":  lambda q: self.get_context(q),
                "question": RunnablePassthrough()
            }
            | self.chat_prompt
            | self.llm
            | StrOutputParser()
        )
        return chain.invoke(question)



Question: What is SearchAgent-X?
Answer: SearchAgent-X is an inference system explicitly designed to optimize end-to-end efficiency for search agent workloads by smoothly interleaving self-reasoning and retrieval. It is a tightly integrated system that processes search agent requests at the token generation level, triggering retrieval actions based on special tags during LLM output steps. SearchAgent-X orchestrates LLM inference with prefix caching, dynamic high-recall approximate retrieval, Priority Scheduling, and Non-Stall Retrieval to achieve efficient search agents. It includes a non-stall retrieval mechanism that allows generation to proceed without unnecessary waiting while ensuring sufficient retrieval quality. Extensive experiments show that SearchAgent-X consistently and significantly outperforms state-of-the-art baseline systems in both offline and online inference scenarios.

Question: What is SearchAgent-X?
Answer: SearchAgent-X is an inference system explicitly designed to optimize end-to-end efficiency for search agent workloads by smoothly interleaving self-reasoning and retrieval. It processes search agent requests at the token generation level, using special tags to trigger ANN-based search or request completion. The system incorporates a priority scheduler to dynamically prioritize concurrent requests based on real-time metrics, enhancing KV-cache reuse and reducing computational overhead. It also employs a non-stall early termination strategy for Approximate Nearest Neighbor (ANN) search to avoid generation stalls by adaptively concluding retrieval based on retrieval result maturity and LLM engine readiness, ensuring efficient and uninterrupted token generation.

In [0]:
rag_fixed = RAGPipeline(llm, chat_prompt, vs_fixed)
query = "What is SearchAgent-X?"
answer = rag_fixed.invoke(query)
print(f"Question: {query}")
print(f"Answer: {answer}")

print()

answer = rag_fixed.invoke_w_lcel(query)
print(f"Question: {query}")
print(f"Answer: {answer}")

Question: What is SearchAgent-X?
Answer: SearchAgent-X is an inference system explicitly designed to optimize end-to-end efficiency for search agent workloads by smoothly interleaving self-reasoning and retrieval. It is a tightly integrated system that processes search agent requests at the token generation level. At each large language model (LLM) output step, the system checks for special tags that trigger the Retriever for an approximate nearest neighbor (ANN)-based search or request completion. SearchAgent-X features a non-stall retrieval mechanism through an adaptive approach that allows generation to proceed without unnecessary waiting while ensuring sufficient retrieval quality. It supports frequent retrieval actions and iteratively incorporates new information into its reasoning process, enabling the search agent to tackle more complex questions and deliver better responses beyond reliance solely on pre-trained knowledge or a single retrieval. Extensive experiments demonstrate 

In [0]:

rag_semantic = RAGPipeline(llm, chat_prompt, vs_semantic)
query = "What is SearchAgent-X?"
answer = rag_semantic.invoke(query)
print(f"Question: {query}")
print(f"Answer: {answer}")

print()

answer = rag_semantic.invoke_w_lcel(query)
print(f"Question: {query}")
print(f"Answer: {answer}")

Question: What is SearchAgent-X?
Answer: SearchAgent-X is a system designed to optimize end-to-end system throughput and latency by smoothly coordinating the interleaving of self-reasoning and retrieval. It autonomously plans retrieval actions and iteratively incorporates new information into its reasoning process, enabling it to tackle more complex questions and deliver better responses beyond relying solely on pre-trained knowledge or a single retrieval. The system orchestrates large language model (LLM) inference with prefix caching, dynamic high-recall approximate retrieval, Priority Scheduling, and Non-Stall Retrieval to achieve efficient search agents. SearchAgent-X manages incoming requests, active asynchronous search tasks, and their results. Extensive experiments show that it consistently and significantly outperforms state-of-the-art baseline systems across various operational settings, achieving substantial improvements in system performance (e.g., 1.3-3.4× higher throughput

In [0]:
rag_semantic = RAGPipeline(llm, chat_prompt, vs_semantic)
query = "What is a Pokemon?"
answer = rag_semantic.invoke(query)
print(f"Question: {query}")
print(f"Answer: {answer}")

print()

Question: What is a Pokemon?
Answer: I do not know



## Insights from the RAG Evaluation

Both retrieval strategies produced answers that were grounded in the source document. The main difference was in the level of detail and narrative focus: fixed chunking emphasized concrete technical components, while semantic chunking provided broader conceptual context.

## Part 2: Adaptive RAG and Databricks Vector Search

This section extends the system into a more production-style architecture by preparing chunks for managed vector indexing and building an adaptive retrieval workflow.

### 1. Data Preparation for Managed Indexing

The retrieval corpus is prepared for Delta synchronization and vector indexing by assigning unique identifiers to each chunk and enabling change tracking.

In [0]:
import os
from typing import TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from databricks.vector_search.client import VectorSearchClient
from dotenv import load_dotenv

load_dotenv("/Workspace/Users/26100305@lums.edu.pk/env")

True

In [0]:
### TO-DO: Prepare the fixed chunks table
spark.sql(f"""
    CREATE OR REPLACE TABLE `{CATALOG}`.processed_data.fixed_chunks_prepared AS
    SELECT
        uuid()    AS Chunk_Id,
        ChunkText AS ChunkText
    FROM `{CATALOG}`.processed_data.fixed_chunks
""")

print(f"fixed_chunks_prepared created")
display(spark.table(f"`{CATALOG}`.processed_data.fixed_chunks_prepared").limit(5))

fixed_chunks_prepared created


Chunk_Id,ChunkText
85c09ad5-772b-44c6-b75e-e33885772bb0,"Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents Tiannuo Yang1*, Zebin Yao1*, Bowen Jin2, Lixiao Cui1, Yusen Li1, Gang Wang1, Xiaoguang Liu1 1 Nankai University 2 University of Illinois Urbana-Champaign {yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn bowenj4@illinois.edu Abstract Large Language Model (LLM)-based search agents have shown remarkable ca- pabilities in solving complex tasks by dynamically decomposing problems and addressing them through interleav"
6429c0a9-08c5-4df1-954d-d382d8a19c9b,"ing problems and addressing them through interleaved reasoning and retrieval. However, this in- terleaved paradigm introduces substantial efficiency bottlenecks. First, we ob- serve that both highly accurate and overly approximate retrieval methods de- grade system efficiency: exact search incurs significant retrieval overhead, while coarse retrieval requires additional reasoning steps during generation. Second, we identify inefficiencies in system design, including improper scheduling and frequent retrieva"
a9833046-6153-4482-921b-96da297c06f7,"ncluding improper scheduling and frequent retrieval stalls, which lead to cascading latency—where even minor de- lays in retrieval amplify end-to-end inference time. To address these challenges, we introduce SearchAgent-X, a high-efficiency inference framework for LLM- based search agents. SearchAgent-X leverages high-recall approximate retrieval and incorporates two key techniques: priority-aware scheduling and non-stall retrieval. Extensive experiments demonstrate that SearchAgent-X consistently outperfor"
b024cb1f-b7f2-486f-8c94-03ccc35add3f,"onstrate that SearchAgent-X consistently outperforms state-of-the-art systems such as vLLM and HNSW-based retrieval across diverse tasks, achieving up to 3.4× higher throughput and 5× lower la- tency, without compromising generation quality. SearchAgent-X is available at https://github.com/tiannuo-yang/SearchAgent-X. 1 Introduction Traditional Retrieval-Augmented Generation (RAG) typically uses a sequential retrieve-then-generate approach [1–8], which limits dynamic interaction with knowledge bases. Recent"
cf913671-e80d-4c8c-a8b0-f04591403d94,"dynamic interaction with knowledge bases. Recent advancements have ushered in RAG 2.0, known as Search Agents [9–16]. This paradigm leverages the strong reasoning capabilities of Large Language Models (LLMs), allowing for the dynamic and adaptive interleaving of reasoning steps with retrieval calls throughout the generation process. Instead of a fixed pipeline, search agents can decide when and what to retrieve based on LLM’s ongoing reasoning, leading to significant improvements in the quality and depth o"


In [0]:
spark.sql(f"""
    ALTER TABLE `{CATALOG}`.processed_data.fixed_chunks_prepared
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

print("CDF enabled")

CDF enabled


### 2. Vector Store Endpoint and Index Management

This stage configures the Databricks vector store environment needed to support managed similarity search at scale.

In [0]:
# FUNCTIONS I MADE IN CASE YOU GUYS NEED TO CLEANUP YOUR WORKSPACE

def vector_store_endpoint_cleanup(vsc):
    endpoints = vsc.list_endpoints().get('endpoints', [])

    if not endpoints:
        print("No endpoints found. Workspace is clear.")
    else:
        for e in endpoints:
            name = e['name']
            print(f"Deleting endpoint: {name}...")
            vsc.delete_endpoint(name)
        
        print("Waiting for workspace quota to refresh...")
        while len(vsc.list_endpoints().get('endpoints', [])) > 0:
            print("Endpoint still terminating... waiting 5s", end="\r")
            time.sleep(5)
        print("\nWorkspace cleared. Quota reset to 0/1.")

def vector_store_index_cleanup(vsc, endpoint_name, index_full_name):
    """Delete the vector store endpoint if it exists"""
    try:
        print(f"Attempting to delete index: {index_full_name}...")
        vsc.delete_index(endpoint_name=endpoint_name, index_name=index_full_name)
        print("Index deletion command sent.")
    except Exception as e:
        print(f"Index not found or already deleted: {e}")

    try:
        print(f"Deleting endpoint: {endpoint_name}...")
        vsc.delete_endpoint(name=endpoint_name)
    except Exception as e:
        print(f"Endpoint not found or already deleted: {e}")

    print("Waiting for Unity Catalog to clear metadata...waiting 5s")
    time.sleep(5)
    print("System cleared. You can now run the creation script.")


In [0]:
os.environ["DATABRICKS_HOST"] = "https://dbc-facb4d62-6b45.cloud.databricks.com"

vsc = VectorSearchClient(
    workspace_url         = "https://dbc-facb4d62-6b45.cloud.databricks.com",
    personal_access_token = os.environ.get("DATABRICKS_TOKEN"),
    disable_notice        = True
)

endpoint_name = "26100305_vector_endpoint"
index_name    = "26100305-pa3.vector_index.fixed_vector_index"

print("VSC initialized with correct host")

VSC initialized with correct host


In [0]:
# Check what host we actually have
host = os.environ.get("DATABRICKS_HOST")
token = os.environ.get("DATABRICKS_TOKEN")

print(f"Host: {host}")
print(f"Token starts with: {token[:10] if token else 'NOT FOUND'}")

Host: https://dbc-facb4d62-6b45.cloud.databricks.com
Token starts with: dapiad49ca


In [0]:
# vector_store_endpoint_cleanup(vsc)
# vector_store_index_cleanup(vsc, endpoint_name, index_name)

In [0]:
print(f"Creating Endpoint: {endpoint_name}...")
### TO-DO: Create the vector store endpoint
# vsc.create_endpoint(
#     name = endpoint_name,
#     endpoint_type = "STANDARD"
# )
print(f"\nCreating index: {index_name}...")

### TO-DO: Create the vector store index
# vsc.create_delta_sync_index(
#     endpoint_name                 = endpoint_name,
#     index_name                    = index_name,
#     source_table_name             = f"26100305-pa3.processed_data.fixed_chunks_prepared",
#     pipeline_type                 = "TRIGGERED",
#     primary_key                   = "Chunk_Id",
#     embedding_source_column       = "ChunkText",
#     embedding_model_endpoint_name = "databricks-gte-large-en"
# )

print("Index creation initiated.")


Creating Endpoint: 26100305_vector_endpoint...

Creating index: 26100305-pa3.vector_index.fixed_vector_index...
Index creation initiated.


### 3. Adaptive Retrieval Workflow

The agent is designed to retrieve context dynamically, evaluate answer relevance, and refine its response when the initial retrieval is insufficient.

In [0]:
llm = AzureChatOpenAI(
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT"), 
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"), 
    temperature=0.0
)

class GraphState(TypedDict):
    question: str
    initial_answer: str
    grade: str
    final_answer: str
    context: str

In [0]:
def retrieve_formatted_context(question: str, topk: int = 3):
    ### TO-DO: Retrieve the top-k most relevant chunks from the vector store
    results = vs_fixed.search_vectors(question, top_k=topk)
    chunks  = [row["Chunk_text"] for row in results.collect()]
    return "\n\n".join(chunks)

### TO-DO
context_retriever_node = RunnableLambda(lambda x: retrieve_formatted_context(x["question"]))

### TO-DO
system_message = SystemMessagePromptTemplate.from_template(
    "You are a precise question-answering, helpful assistant. Your ONLY source of truth is the context below.\n\n"
    "Rules you must follow without exception:\n"
    "1. Answer ONLY using information explicitly stated in the context.\n"
    "2. Do NOT use your general knowledge or make inferences beyond the context.\n"
    "3. If the context does not contain enough information to answer, respond EXACTLY with: 'I do not know'\n"
    "4. Do not make up facts, figures, or details.\n"
    "5. Provide a thorough and detailed answer using ALL relevant information from the context.\n\n"
    "Context:\n{context}"
)

### TO-DO
human_message = HumanMessagePromptTemplate.from_template("{question}")

### TO-DO
chat_prompt = ChatPromptTemplate.from_messages([system_message, human_message])

### TO-DO
rag_chain = (
    {
        "context":  context_retriever_node,
        "question": RunnablePassthrough()
    }
    | chat_prompt
    | llm
    | StrOutputParser()
)

def base_rag_node(state: GraphState) -> dict:
    ### TO-DO
    question = state["question"]
    initial_answer = rag_chain.invoke({"question": question})
    return {"initial_answer": initial_answer}



In [0]:
### TO-DO
class AnswerGrader(BaseModel):
    binary_score: str = Field(description="Answer is relevant to the question? 'yes' or 'no'")

### TO-DO
grader_system_message = SystemMessagePromptTemplate.from_template(
    "You are a strict relevance grader evaluating whether an AI-generated answer "
    "adequately addresses the user's question.\n\n"
    "Grading criteria:\n"
    "- Score 'yes' if: the answer directly addresses the question with specific, relevant information\n"
    "- Score 'no' if: the answer says 'I do not know', is vague, off-topic, or lacks substance\n\n"
    "You must respond with ONLY a binary score — 'yes' or 'no'. No explanation."
)

### TO-DO
grader_human_message = HumanMessagePromptTemplate.from_template(
    "Question: {question}\n\n"
    "Answer: {initial_answer}\n\n"
    "Is this answer relevant and sufficient? Respond with 'yes' or 'no'."
)

### TO-DO
grader_prompt = ChatPromptTemplate.from_messages([
    grader_system_message,
    grader_human_message
])

### TO-DO
grader_chain = grader_prompt | llm.with_structured_output(AnswerGrader)


def grader_node(state: GraphState) -> dict:
    ### TO-DO
    question       = state["question"]
    initial_answer = state["initial_answer"]
    result         = grader_chain.invoke({
        "question":       question,
        "initial_answer": initial_answer
    })
    return {"grade": result.binary_score}



In [0]:
### TO-DO
class CategoryRouter(BaseModel):
    category: str = Field(description="Category of the question: 'azure' or 'databricks'")

### TO-DO
router_prompt = ChatPromptTemplate.from_template(
    "You are a query classifier. Your job is to classify the question below into "
    "exactly one of two categories:\n\n"
    "'azure' — if the question is about Microsoft Azure, Azure services, Azure OpenAI, "
    "Azure Data Factory, Azure Synapse, ADLS, Azure networking, or any Microsoft cloud topic.\n\n"
    "'databricks' — if the question is about Databricks, Delta Lake, Unity Catalog, "
    "Apache Spark, MLflow, Databricks Workflows, Vector Search, or the Lakehouse architecture.\n\n"
    "Rules:\n"
    "1. Respond with ONLY the single word 'azure' or 'databricks' — nothing else.\n"
    "2. Do not explain your reasoning.\n"
    "3. If the question could fit both, pick the most dominant topic.\n"
    "4. If completely unrelated to either, default to 'databricks'.\n\n"
    "Question: {question}"
)

### TO-DO
router_chain = router_prompt | llm.with_structured_output(CategoryRouter)


def text_file_fallback_node(state: GraphState) -> dict:
    ### TO-DO
    question = state["question"]

    result   = router_chain.invoke({"question": question})
    category = result.category.lower()

    file_map = {
        "databricks": f"/Volumes/{CATALOG}/default/text_documents/databricks_info.txt",
        "azure":      f"/Volumes/{CATALOG}/default/text_documents/azure_info.txt",
    }

    file_path = file_map.get(category, None)
    if file_path is None:
        return {"final_answer": f"Category not found: {category}"}

    try:
        with open(file_path, "r") as f:
            full_text = f.read()

        response = llm.invoke(
            f"Answer this question using ONLY the text below.\n\n"
            f"Question: {question}\n\n"
            f"Text:\n{full_text}"
        )
        return {"final_answer": response.content}

    except Exception as e:
        return {"final_answer": f"Fallback failed. Could not read file at {file_path}. Error: {e}"}


In [0]:
test_state = {"question": "What is SearchAgent-X?", "initial_answer": "", "grade": "", "final_answer": "", "context": ""}
result = base_rag_node(test_state)
print(result["initial_answer"])

SearchAgent-X is an inference system explicitly designed to optimize end-to-end efficiency for search agent workloads by smoothly interleaving self-reasoning and retrieval. It is a tightly integrated system that processes search agent requests at the token generation level. At each large language model (LLM) output step, the system checks for special tags that trigger the Retriever for an approximate nearest neighbor (ANN)-based search or request completion. SearchAgent-X features a non-stall retrieval mechanism through an adaptive approach that allows generation to proceed without unnecessary waiting while ensuring sufficient retrieval quality. It enables frequent retrieval actions and iterative incorporation of new information into the reasoning process, allowing the search agent to tackle more complex questions and deliver better responses beyond reliance solely on pre-trained knowledge or a single retrieval. Extensive experiments demonstrate that SearchAgent-X consistently and sign

In [0]:
test_state["initial_answer"] = result["initial_answer"]
grade_result = grader_node(test_state)
print(f"Grade: {grade_result['grade']}")

Grade: yes


In [0]:
fallback_state = {"question": "What is Azure Key Vault?", "initial_answer": "I do not know", "grade": "no", "final_answer": "", "context": ""}
fallback_result = text_file_fallback_node(fallback_state)
print(fallback_result["final_answer"])

Azure Key Vault is a service in Azure that securely stores sensitive credentials like API keys.


### 4. Building the Agentic Graph

The workflow is expressed as a stateful graph so the system can route through retrieval, grading, and fallback behavior in a structured manner.

In [0]:

### TO-DO: Using these functions, create a state graph that will answer, grade, reanswer a questions
def route_based_on_grade(state: GraphState) -> str:
    if state["grade"] == "yes":
        return END
    else:
        return "file_fallback_node"

workflow = StateGraph(GraphState)

workflow.add_node("local_rag_node", base_rag_node)
workflow.add_node("grader_node", grader_node)
workflow.add_node("file_fallback_node", text_file_fallback_node)

workflow.add_edge(START, "local_rag_node")
workflow.add_edge("local_rag_node", "grader_node")

workflow.add_conditional_edges(
    "grader_node",
    route_based_on_grade,
    {
        END: END,
        "file_fallback_node": "file_fallback_node"
    }
)
compiled_adaptive_rag = workflow.compile()

### 5. Running the Agentic Workflow

This section executes the adaptive retrieval flow and demonstrates how the system responds to user queries using the constructed agent graph.

In [0]:
inputs = {"question": "What is Databricks?"}
final_state = None

print(f"--- STARTING ADAPTIVE RAG FLOW ---")
print(f"User Question: {inputs['question']}\n")

### TO-DO
def agentic_rag_flow(question):
    print(f"--- STARTING ADAPTIVE RAG FLOW ---")
    print(f"User Question: {question}\n")
    
    full_state = {}
    
    for step in compiled_adaptive_rag.stream({"question": question}):
        for node_name, state_update in step.items():
            print(f"--- Active Node: [{node_name}] ---")
            
            full_state.update(state_update)
            
            for key, value in state_update.items():
                if value:
                    print(f"  {key}: {str(value)[:300]}")
            print()

    final_answer = full_state.get("final_answer") or full_state.get("initial_answer", "No answer found")
    
    print(f"--- FINAL RESOLVED ANSWER ---")
    print(final_answer)
    return final_answer

agentic_rag_flow(inputs["question"])

--- STARTING ADAPTIVE RAG FLOW ---
User Question: What is Databricks?

--- STARTING ADAPTIVE RAG FLOW ---
User Question: What is Databricks?

--- Active Node: [local_rag_node] ---
  initial_answer: I do not know

--- Active Node: [grader_node] ---
  grade: no

--- Active Node: [file_fallback_node] ---
  final_answer: Databricks is the pioneer of the Lakehouse Architecture, a unified platform that combines the best elements of data lakes and data warehouses. Built on top of Apache Spark, Delta Lake, and MLflow, it provides a collaborative environment for data engineers, data scientists, and analysts. By 2026, it 

--- FINAL RESOLVED ANSWER ---
Databricks is the pioneer of the Lakehouse Architecture, a unified platform that combines the best elements of data lakes and data warehouses. Built on top of Apache Spark, Delta Lake, and MLflow, it provides a collaborative environment for data engineers, data scientists, and analysts. By 2026, it has evolved into a "Data Intelligence Platform" t

'Databricks is the pioneer of the Lakehouse Architecture, a unified platform that combines the best elements of data lakes and data warehouses. Built on top of Apache Spark, Delta Lake, and MLflow, it provides a collaborative environment for data engineers, data scientists, and analysts. By 2026, it has evolved into a "Data Intelligence Platform" that uses generative AI to simplify data management and governance.'

In [0]:
# Check endpoint status
endpoint = vsc.get_endpoint(name=endpoint_name)
print(f"Endpoint status: {endpoint['endpoint_status']['state']}")

# Check index status
index = vsc.get_index(
    endpoint_name = endpoint_name,
    index_name    = index_name
)
# Use describe() to get status dict
index_info = index.describe()
print(f"Index status: {index_info['status']['ready']}")
print(f"Index state:  {index_info['status'].get('index_url', 'still syncing...')}")

Endpoint status: ONLINE
Index status: True
Index state:  dbc-facb4d62-6b45.cloud.databricks.com/api/2.0/vector-search/indexes/26100305-pa3.vector_index.fixed_vector_index


## MCP-Based Tool Integration

The final section extends the workflow with an MCP server interface, allowing the agent to invoke tools through a standardized protocol and interact with retrieval functionality in a modular way.

In [0]:
%pip install langchain langchain-openai langchain-core langgraph mlflow databricks-sdk

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# %pip install --upgrade mlflow[databricks] databricks-agents
# dbutils.library.restartPython()

In [0]:
import os
import time
import yaml
import mlflow
from dotenv import load_dotenv
from mlflow.pyfunc import ResponsesAgent
from typing import Any, Dict
from langchain_openai import AzureChatOpenAI
from langchain.tools import tool
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from databricks.vector_search.client import VectorSearchClient

load_dotenv("/Workspace/Users/26100305@lums.edu.pk/env")


True

In [0]:
config = {
    "llm_endpoint": "lums-gpt-4.1-mini",
    "mcp_servers": [
        {
            "name": "databricks_managed_mcp",
            "url": "https://dbc-facb4d62-6b45.cloud.databricks.com/api/2.0/mcp/functions/system/ai"
        }
    ]
}

with open("/Workspace/Users/26100305@lums.edu.pk/agent_config.yml", "w") as f:
    yaml.dump(config, f)

print("agent_config.yml created")

agent_config.yml created


In [0]:
@tool
def vector_search_tool(query: str) -> str:
    """
    Searches the vector knowledge base for relevant document chunks based on a query.
    Use this tool whenever you need to find information about RAG systems, SearchAgent-X,
    LLM efficiency, retrieval mechanisms, or any topic from the research paper.
    Always use this tool before answering — do not rely on your own knowledge.

    Args:
        query: The search query string.
    Returns:
        Relevant document chunks as a formatted string.
    """
    try:
        results = vs_fixed.search_vectors(query, top_k=5)
        chunks  = [row["Chunk_text"] for row in results.collect()]
        return "\n\n".join(chunks) if chunks else "No relevant documents found."
    except Exception as e:
        return f"Database search failed: {e}"

def vector_store_endpoint_cleanup(vsc):
    ### TO-DO: Implement a function that deletes the vector store endpoint if they exist and write a detailed docstring for the function
    endpoints = vsc.list_endpoints().get('endpoints', [])
    if not endpoints:
        print("No endpoints found. Workspace is clear.")
    else:
        for e in endpoints:
            name = e['name']
            print(f"Deleting endpoint: {name}...")
            vsc.delete_endpoint(name)
        print("Waiting for workspace quota to refresh...")
        while len(vsc.list_endpoints().get('endpoints', [])) > 0:
            print("Endpoint still terminating... waiting 5s", end="\r")
            time.sleep(5)
        print("\nWorkspace cleared. Quota reset to 0/1.")



def vector_store_index_cleanup(vsc, endpoint_name, index_full_name):
    ### TO-DO: Implement a function that deletes the vector store index if they exist and write a detailed docstring for the function
    try:
        print(f"Attempting to delete index: {index_full_name}...")
        vsc.delete_index(endpoint_name=endpoint_name, index_name=index_full_name)
        print("Index deletion sent.")
    except Exception as e:
        print(f"Index not found or already deleted: {e}")
    try:
        print(f"Deleting endpoint: {endpoint_name}...")
        vsc.delete_endpoint(name=endpoint_name)
    except Exception as e:
        print(f"Endpoint not found or already deleted: {e}")
    print("Waiting for Unity Catalog to clear metadata...waiting 5s")
    time.sleep(5)
    print("System cleared. creation script can run now.")


In [0]:

class VectorSearchAgent(ResponsesAgent):
    def __init__(self, config_path: str):
        with open(config_path, 'r') as f:
            self.config = yaml.safe_load(f)
        super().__init__()
        
        self.system_prompt = (
            "You are a precise data engineering assistant. "
            "To answer questions about the course or documentation, you MUST use "
            "the provided vector_search_tool to retrieve context. "
            "If the tool does not return relevant information, state that you do not know."
        )

    def predict(self, context, payload) -> Dict[str, Any]:
        """Executes the Agent Loop locally."""
        if isinstance(payload, dict):
            messages = payload.get("messages", payload.get("input", []))
        elif isinstance(payload, list):
            messages = payload
        else:
            messages = getattr(payload, "messages", getattr(payload, "input", []))
            
        user_question = messages[-1]['content'] if isinstance(messages[-1], dict) else messages[-1].content
        
        ### TO-DO
        llm = AzureChatOpenAI(
            azure_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT"),
            azure_deployment = os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT"),
            api_version = os.environ.get("AZURE_OPENAI_API_VERSION"),
            api_key = os.environ.get("AZURE_OPENAI_API_KEY"),
            temperature = 0.0
        )
        
        prompt_template = ChatPromptTemplate.from_messages([
            ("system", self.system_prompt),
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad")
        ])
        
        tools= [vector_search_tool]
        agent_logic  = create_tool_calling_agent(llm, tools, prompt_template)
        agent_executor = AgentExecutor(
            agent = agent_logic,
            tools = tools,
            verbose = True
        )
        
        # RUN THE LOOP
        raw_response = agent_executor.invoke({"input": user_question})["output"]
        
        return {"choices": [{"message": {"role": "assistant", "content": raw_response}}]}

agent = VectorSearchAgent(config_path="agent_config.yml")
mlflow.models.set_model(model=agent)


In [0]:
# testing
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

test_payload = {
    "messages": [{"role": "user", "content": "What is SearchAgent-X and how does it improve efficiency?"}]
}

response = agent.predict(context=None, payload=test_payload)
print("=== AGENT RESPONSE ===")
print(response["choices"][0]["message"]["content"])



> Entering new AgentExecutor chain...

Invoking: `vector_search_tool` with `{'query': 'SearchAgent-X efficiency improvement'}`


Agent-X utilizes computing resources more efficiently than baselines,
completing more requests in the same amount of time. As shown in Figure 5, SearchAgent-X
completes at least 1.5×, and up to 3.5× more requests on average than the baseline, within the
request rate range of 1 to 6. We also observe that the advantage of SearchAgent-X over the
baselines increases with the request rate. For example, at a request rate of 6, SearchAgent-X
achieves 5.8× more requests than vLLM_ENN, and 1.9× more than the most co

e frequent retrieval stalls, SearchAgent-X proposes a non-stall retrieval mechanism through an adaptive mechanism that allows generation to proceed without unnecessary waiting while ensuring sufficient retrieval quality. Our extensive experiments demonstrate that SearchAgent-X consistently and significantly outper-
forms state-of-the-art baseline system

Trace(trace_id=tr-08ce07b1ce1722742d728a0d57c1958e)

### Evaluation and Testing

The agent is tested with a range of questions to validate both its retrieval behavior and its ability to use MCP-based tools effectively.

In [0]:
test_questions = [
    "What is the Databricks Vector Search endpoint cleanup function supposed to do?",
    "What tools do you have and what do they do?"
]

print("Starting Local Agent Executor Test...\n" + "="*50)

for question in test_questions:
    print(f"\nUser Query: {question}")
    
    payload = {
        "input": [{"role": "user", "content": question}],
        "messages": [{"role": "user", "content": question}]
    }
    
    # Invoke the agent — use context=None
    response     = agent.predict(context=None, payload=payload)
    final_answer = response["choices"][0]["message"]["content"]
    
    print(f"Final Answer:\n{final_answer}")
    print("-" * 50)

Starting Local Agent Executor Test...

User Query: What is the Databricks Vector Search endpoint cleanup function supposed to do?


> Entering new AgentExecutor chain...

Invoking: `vector_search_tool` with `{'query': 'Databricks Vector Search endpoint cleanup function'}`


quest completion (e.g., <anser>), respectively. To optimize GPU resource usage, SearchAgent-X incorporates a priority scheduler. It dynamically prioritizes concurrent requests using real-time collected metrics like retrieval count and waiting time, aiming to enhance KV-cache reuse by processing higher-priority requests first. During prefill, prefix matching reuses existing KV pairs from cache, significantly reducing computational overhead; new KV states are computed if caching is inapplicable or a miss occu

 from drastic efficiency degradation under even minor retrieval delays. As average retrieval latency increases from 0.6s to 4.4s, the end-to-end latency of the search agent is magnified by over 83×, while RAG re

[Trace(trace_id=tr-3efeae21165712f52b10e2b4b6d95763), Trace(trace_id=tr-d931dbf6232267a03dbf70bed7ec0e1e)]

## Project Summary

This notebook showcases a complete end-to-end prototype for agentic RAG on Databricks, combining document ingestion, chunking, embeddings, vector retrieval, LangGraph orchestration, and MCP tool integration. The system demonstrates a practical architecture for grounding LLM-based agents in domain-specific knowledge while maintaining modular, extensible behavior.